# BTFR Analysis: Complete Recomputation from SPARC Raw Data
Computes Vout from rotation curves, merges with SPARC table,
and compares Vflat vs Vout in the Baryonic Tully-Fisher Relation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import glob
import os

## Step 1: Compute Vout from rotation curves

In [ ]:
data_path = '/content/drive/MyDrive/SPARC_rotmod_files/'
files = sorted(glob.glob(data_path + '*.dat'))
print(f'Found files: {len(files)}')

vout_list = []

for f in files:
    galaxy = os.path.basename(f).replace('_rotmod.dat', '')
    try:
        dat = pd.read_csv(f, sep=r'\s+', comment='#',
                          names=['r','Vobs','errV','Vgas',
                                 'Vdisk','Vbul','SBdisk','SBbul'])
        dat = dat[dat['Vobs'] > 0].dropna()
        if len(dat) < 3:
            continue
        last3 = dat.tail(3)
        vout_list.append({
            'galaxy':   galaxy,
            'Vout':     last3['Vobs'].median(),
            'Verr':     last3['errV'].median(),
            'n_points': len(dat)
        })
    except Exception as e:
        print(f'Error at {galaxy}: {e}')

vout_df = pd.DataFrame(vout_list)
print(f'Vout computed for {len(vout_df)} galaxies')

## Step 2: Load SPARC table

In [ ]:
sparc = pd.read_fwf(
    '/content/SPARC_Lelli2016c.mrt',
    skiprows=98,
    names=['Galaxy','T','D','e_D','f_D',
           'Inc','e_Inc','L36','e_L36',
           'Reff','SBeff','Rdisk','SBdisk',
           'MHI','RHI','Vflat','e_Vflat',
           'Q','Ref'])
sparc['galaxy'] = sparc['Galaxy'].str.strip()
print(f'SPARC table loaded: {len(sparc)} galaxies')

## Step 3: Compute baryonic mass
Upsilon = 0.5 M_sun/L_sun at 3.6 µm (McGaugh & Schombert 2015)

In [ ]:
sparc['Mstar'] = 0.5  * sparc['L36']   # 1e9 M_sun
sparc['Mgas']  = 1.33 * sparc['MHI']   # He correction
sparc['Mbar']  = sparc['Mstar'] + sparc['Mgas']

## Step 4: Merge and clean

In [ ]:
merged = pd.merge(vout_df, sparc, on='galaxy', how='inner')
print(f'After merge: {len(merged)} galaxies')

clean = merged[
    (merged['Vflat'] > 0) &
    (merged['Vout']  > 0) &
    (merged['Mbar']  > 0) &
    (merged['Q']     <= 2)
].copy()
print(f'After quality filter (Q<=2, Vflat>0): {len(clean)} galaxies')

excluded = merged[(merged['Vflat'] == 0) | (merged['Q'] > 2)]
print(f'Excluded (Vflat=0 or Q>2): {len(excluded)} galaxies')

clean['logV_flat'] = np.log10(clean['Vflat'])
clean['logV_out']  = np.log10(clean['Vout'])
clean['logM']      = np.log10(clean['Mbar'] * 1e9)

## Step 5: BTFR fit and residuals

In [ ]:
def compute_residuals(df, vcol):
    sl, ic, r, p, se = stats.linregress(df[vcol], df['logM'])
    res = df['logM'] - (sl * df[vcol] + ic)
    return res, sl, ic, r, p

clean['res_flat'], sl_f, ic_f, r_f, p_f = \
    compute_residuals(clean, 'logV_flat')
clean['res_out'],  sl_o, ic_o, r_o, p_o = \
    compute_residuals(clean, 'logV_out')

rms_flat = np.std(clean['res_flat'])
rms_out  = np.std(clean['res_out'])
red_pct  = (rms_flat - rms_out) / rms_flat * 100

print(f'=== BTFR Results ===')
print(f'n galaxies:       {len(clean)}')
print(f'Slope Vflat:      {sl_f:.3f}')
print(f'Slope Vout:       {sl_o:.3f}')
print(f'RMS Vflat:        {rms_flat:.4f} dex')
print(f'RMS Vout:         {rms_out:.4f} dex')
print(f'Global reduction: {red_pct:.1f}%')

median_logV = clean['logV_flat'].median()
for label, sub in [('High-acc', clean[clean['logV_flat'] >= median_logV]),
                   ('Low-acc',  clean[clean['logV_flat'] <  median_logV])]:
    rf = np.std(sub['res_flat'])
    ro = np.std(sub['res_out'])
    print(f'{label} (n={len(sub)}): '
          f'RMS_flat={rf:.4f}, RMS_out={ro:.4f}, '
          f'reduction={((rf-ro)/rf*100):.1f}%')

improvement = clean['res_flat'].abs() - clean['res_out'].abs()
r_imp, p_imp = stats.pearsonr(clean['logV_flat'], improvement)
print(f'Correlation improvement vs logV: r={r_imp:.3f}, p={p_imp:.4f}')

## Step 6: Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(clean['logV_flat'], clean['logM'],
           s=20, alpha=0.6, color='steelblue',
           label=f'$V_\\mathrm{{flat}}$ '
                 f'($\\sigma={rms_flat:.3f}$ dex)')
ax.scatter(clean['logV_out'], clean['logM'],
           s=20, alpha=0.6, color='darkorange',
           label=f'$V_\\mathrm{{out}}$ '
                 f'($\\sigma={rms_out:.3f}$ dex)')
x_range = np.linspace(clean['logV_flat'].min()-0.1,
                      clean['logV_flat'].max()+0.1, 100)
ax.plot(x_range, sl_f*x_range+ic_f, 'b--', linewidth=1.5)
ax.plot(x_range, sl_o*x_range+ic_o, '-',
        color='darkorange', linewidth=1.5)
ax.set_xlabel('$\\log_{10}(V\\ [\\mathrm{km/s}])$', fontsize=12)
ax.set_ylabel('$\\log_{10}(M_\\mathrm{bar}/M_\\odot)$', fontsize=12)
ax.set_title('Baryonic Tully-Fisher Relation', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

ax = axes[1]
ax.scatter(clean['logV_flat'], improvement,
           s=20, alpha=0.6, color='steelblue')
ax.axhline(0, linestyle='--', color='black',
           linewidth=1.5, alpha=0.7, label='No improvement')
ax.axvline(median_logV, linestyle=':',
           color='red', linewidth=1.2, alpha=0.7,
           label=f'Median $\\log V = {median_logV:.2f}$')
ax.set_xlabel('$\\log_{10}(V_\\mathrm{flat})$ — '
              'proxy for acceleration', fontsize=12)
ax.set_ylabel('Improvement\n'
              '($|\\mathrm{res}_\\mathrm{flat}| - '
              '|\\mathrm{res}_\\mathrm{out}|$)', fontsize=12)
ax.set_title(f'Improvement vs. Acceleration\n'
             f'$r={r_imp:.3f}$, $p={p_imp:.4f}$', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('/content/btfr_comparison.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Saved: btfr_comparison.png')

## Step 7: Save results

In [ ]:
clean.to_csv('/content/clean_btfr.csv', index=False)
merged.to_csv('/content/merged_btfr.csv', index=False)
print('Saved: clean_btfr.csv')
print('Saved: merged_btfr.csv')